# 058 — Autoencoders, GAN y difusión

Este notebook reutiliza el mismo núcleo ejecutable que `lab.py`. El objetivo no es
ocultar la implementación, sino separar exploración, ejercicio y solución.

**Evidencia esperada:** resultado JSON, interpretación de una decisión y una
limitación documentada.


In [ ]:
from ai_evolution.labs import run_lab
import json

def show(value):
    print(json.dumps(value, ensure_ascii=False, indent=2))


## Solución explicada

**Ejercicio 1.** (a) z = 0.8·8+0.6·6 = 10; x̂ = (8, 6); error = **0** (x es paralelo
a w). (b) z = 0.8·6+0.6·8 = 9.6; x̂ = (7.68, 5.76); error = (6−7.68)² + (8−5.76)² =
2.8224 + 5.0176 = **7.84**. El autoencoder lineal solo conserva la proyección sobre w:
la componente ortogonal (aquí grande) se pierde — exactamente PCA a 1 componente.

**Ejercicio 2.** ᾱ=0.99: x_t = 0.995·2 + 0.1·(−0.5) = **1.9400**. ᾱ=0.5:
0.7071·2 − 0.3536 = **1.0607**. ᾱ=0.01: 0.1·2 − 0.4975 = **−0.2975**. Los regímenes
extremos son fáciles (casi todo señal / casi todo ruido conocido en distribución); el
intermedio, donde señal y ruido pesan parecido, es donde la predicción de ε exige
entender la estructura de los datos.

**Ejercicio 3.** z = μ + σ·ε = (0.5, 1.0, 2.0). El azar queda encapsulado en ε
(constante respecto a los parámetros): z es función determinista y diferenciable de
μ y σ, así que ∂z/∂μ = 1 y ∂z/∂σ = ε existen y el gradiente del ELBO fluye.

**Ejercicio 4.** **Mode collapse**: nitidez alta, diversidad rota (2 de 10 modos).
La pérdida ≈ log 2 solo dice que D no distingue *las muestras que G produce*; no mide
si G cubre todos los modos. Por eso la evaluación generativa requiere métricas de
diversidad (FID, recall) e inspección humana.


In [ ]:
result = run_lab("generation", seed=58)
assert result["kind"] == "generation"
assert result["evidence"]
show(result)


In [ ]:
# Verificación numérica
import math

# Ejercicio 1
w = (0.8, 0.6)
for x in [(8.0, 6.0), (6.0, 8.0)]:
    z = w[0]*x[0] + w[1]*x[1]
    x_hat = (z*w[0], z*w[1])
    err = (x[0]-x_hat[0])**2 + (x[1]-x_hat[1])**2
    print(f"x={x}  z={z}  x̂=({x_hat[0]:.2f}, {x_hat[1]:.2f})  err={err:.4f}")

# Ejercicio 2
x0, eps = 2.0, -0.5
for abar in (0.99, 0.5, 0.01):
    x_t = math.sqrt(abar)*x0 + math.sqrt(1-abar)*eps
    print(f"ᾱ={abar}: x_t={x_t:.4f}")

# Ejercicio 3
mu, sigma = 1.0, 0.5
zs = [mu + sigma*e for e in (-1.0, 0.0, 2.0)]
print("z =", zs)
assert zs == [0.5, 1.0, 2.0]


## Reflexión

1. ¿Qué aporta exactamente el término KL del VAE que hace muestreable su latente, y qué coste tiene sobre la fidelidad de reconstrucción?
2. ¿Por qué la pérdida de difusión (regresión de ruido) es estable donde el minimax de la GAN oscila?
3. Si necesitas generar 10 000 imágenes por segundo con calidad aceptable, ¿qué familia eliges y qué sacrificas?
